### Configuração inicial
Define o caminho onde estão os arquivos de referência usados pelo projeto.

In [0]:
from pyspark.sql import functions as F

CAMINHO = "/Volumes/voebem/bronze/arquivos/referencias"

### Leitura dos aeródromos
Lê o arquivo de aeródromos públicos da ANAC, configurando o separador, cabeçalho, encoding e tratamento das linhas do CSV.

In [0]:
# Leitura do arquivo CSV com inferência automática de schema, header e separador
df_aerodromos = (
    spark.read.format("csv")                   # Corrigido: sem espaços vazios
    .option("sep", ";")                        # Define o ponto-e-vírgula como separador
    .option("header", "true")                  # A primeira linha (após o salto) será o cabeçalho
    .option("skipRows", 1)                     # Ignora a primeira linha inútil do arquivo [cite: 2.1.2]
    .option("quote", "\"")                     # Define as aspas duplas como bloqueador de texto
    .option("escape", "\"")                    # Permite o uso de aspas dentro do próprio texto
    .option("encoding", "ISO-8859-1")               # Garante que palavras com acento não quebrem
    .option("mode", "PERMISSIVE")              # Comportamento padrão: ignora erros de formatação em linhas corrompidas
    .load(f"{CAMINHO}/AerodromosPublicos.csv") # Lembre-se de apontar para o arquivo em si (e não só para a pasta)
)

### Leitura das empresas aéreas
Lê as bases de empresas aéreas nacionais e estrangeiras que serão usadas como referências no projeto.

In [0]:
# Leitura do arquivo CSV de empresas aéreas estrangeiras
df_empresas_estrangeiras = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("quote", "\"")
    .option("escape", "\"")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(f"{CAMINHO}/pda_empresas_aereas_estrangeiros.csv")
)

# Leitura do arquivo CSV de empresas aéreas nacionais
df_empresas_nacionais = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("quote", "\"")
    .option("escape", "\"")
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(f"{CAMINHO}/pda_empresas_aereas_nacionais.csv")
)

### Padronização das colunas
Normaliza os nomes das colunas, removendo acentos e caracteres especiais e deixando os nomes em um padrão mais simples para uso com Spark e SQL.

In [0]:
# Padronização dos nomes das colunas: remove acentos, espaços e caracteres especiais
import unicodedata
import re

def normalizar_nome_coluna(nome):
    """Remove acentos, substitui espaços por underscore e remove caracteres especiais"""
    # Remove acentos
    nome = unicodedata.normalize('NFKD', nome).encode('ASCII', 'ignore').decode('ASCII')
    # Substitui espaços por underscore
    nome = nome.replace(' ', '_')
    # Remove caracteres especiais, mantém apenas letras, números e underscore
    nome = re.sub(r'[^a-zA-Z0-9_]', '', nome)
    # Converte para minúsculas
    nome = nome.lower()
    return nome

# Aplica a normalização em todas as colunas
df_aerodromos = df_aerodromos.toDF(*[normalizar_nome_coluna(col) for col in df_aerodromos.columns])
df_empresas_estrangeiras = df_empresas_estrangeiras.toDF(*[normalizar_nome_coluna(col) for col in df_empresas_estrangeiras.columns])
df_empresas_nacionais = df_empresas_nacionais.toDF(*[normalizar_nome_coluna(col) for col in df_empresas_nacionais.columns])

### Metadados de ingestão
Adiciona o nome do arquivo de origem e o horário da ingestão para facilitar o rastreamento dos dados.

In [0]:
from pyspark.sql import functions as F

# Adiciona colunas de metadata em cada DataFrame
# _arquivo_origem: nome do arquivo de origem
# _ingerido_em: timestamp de ingestão (horário atual)

df_aerodromos = df_aerodromos.withColumn("_arquivo_origem", F.col("_metadata.file_name")) \
    .withColumn("_ingerido_em", F.current_timestamp())

df_empresas_estrangeiras = df_empresas_estrangeiras.withColumn("_arquivo_origem", F.col("_metadata.file_name")) \
    .withColumn("_ingerido_em", F.current_timestamp())

df_empresas_nacionais = df_empresas_nacionais.withColumn("_arquivo_origem", F.col("_metadata.file_name")) \
    .withColumn("_ingerido_em", F.current_timestamp())

### Gravação das tabelas Bronze
Salva as três bases de referência como tabelas Delta no schema `voebem.bronze`, substituindo a versão anterior a cada execução.

In [0]:
# Escrita dos DataFrames em formato Delta com overwrite

# Tabela de aeródromos
df_aerodromos.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("voebem.bronze.aerodromos")

# Tabela de empresas estrangeiras
df_empresas_estrangeiras.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("voebem.bronze.empresas_estrangeiras")

# Tabela de empresas nacionais
df_empresas_nacionais.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("voebem.bronze.empresas_nacionais")

### Conferência dos dados
Exibe algumas linhas de cada tabela para verificar se os dados foram carregados corretamente.

In [0]:
# Consulta SQL para retornar 5 linhas de cada uma das 3 tabelas

# Aeródromos
print("\n=== AERÓDROMOS (5 linhas) ===")
display(spark.sql("""
    SELECT *
    FROM voebem.bronze.aerodromos
    LIMIT 5
"""))

# Empresas Estrangeiras
print("\n=== EMPRESAS ESTRANGEIRAS (5 linhas) ===")
display(spark.sql("""
    SELECT *
    FROM voebem.bronze.empresas_estrangeiras
    LIMIT 5
"""))

# Empresas Nacionais
print("\n=== EMPRESAS NACIONAIS (5 linhas) ===")
display(spark.sql("""
    SELECT *
    FROM voebem.bronze.empresas_nacionais
    LIMIT 5
"""))

### Criação do dicionário de códigos
Cria uma pequena tabela de referência com os códigos de operação e de tipo de linha utilizados nos dados da VRA.

In [0]:
# 1. Os dados da sua imagem
dados_dicionario = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira")
]

colunas = ["dominio", "codigo", "descricao"]

# 2. Criamos o DataFrame
df_dicionario_anac = spark.createDataFrame(dados_dicionario, colunas)

# 3. Salvando como Tabela Delta
(
    df_dicionario_anac.write
    .format("delta")         # Define explicitamente que o formato é Delta Lake
    .mode("overwrite")       # Se a tabela já existir, ele sobrescreve com a versão nova
    .saveAsTable("voebem.bronze.codigos_operacao") # Salva no banco de dados 'default'
)

### Validação da quantidade de registros
Confere a quantidade de registros carregados em cada uma das tabelas principais de referência.

In [0]:
%sql
SELECT 'aerodromos' AS tabela, COUNT(*) AS total_registros FROM voebem.bronze.aerodromos
UNION ALL
SELECT 'empresas_estrangeiras' AS tabela, COUNT(*) AS total_registros FROM voebem.bronze.empresas_estrangeiras
UNION ALL
SELECT 'empresas_nacionais' AS tabela, COUNT(*) AS total_registros FROM voebem.bronze.empresas_nacionais

### Conferência do schema de aeródromos
Exibe a estrutura da tabela de aeródromos.

In [0]:
%sql
DESCRIBE voebem.bronze.aerodromos;


### Conferência do schema de empresas estrangeiras
Exibe a estrutura da tabela de empresas aéreas estrangeiras.

In [0]:
%sql
DESCRIBE voebem.bronze.empresas_estrangeiras;

### Conferência do schema de empresas nacionais
Exibe a estrutura da tabela de empresas aéreas nacionais.

In [0]:
%sql
DESCRIBE voebem.bronze.empresas_nacionais;

### Conferência do schema de códigos
Exibe a estrutura da tabela de códigos de operação.

In [0]:
%sql
DESCRIBE voebem.bronze.codigos_operacao;

### Conferência final
Exibe novamente algumas linhas das tabelas principais para uma conferência final dos dados carregados.

In [0]:
print("Aeródromos:")
display(spark.sql("SELECT * FROM voebem.bronze.aerodromos LIMIT 5"))

print("Empresas Estrangeiras:")
display(spark.sql("SELECT * FROM voebem.bronze.empresas_estrangeiras LIMIT 5"))

print("Empresas Nacionais:")
display(spark.sql("SELECT * FROM voebem.bronze.empresas_nacionais LIMIT 5"))